# A LangGraph Message Graph Example

Now we will make our efforts a bit more agentic. We will add an LLM to facilitate a conversation.

In [ ]:
import os
from typing import List
from IPython.display import display
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
import dotenv

dotenv.load_dotenv()
print("Environment variables loaded.")
print(os.environ["LANGCHAIN_CHAT_MODEL_OLLAMA"])

We create a state with a `messages` field. This is a list we will populate with LLM messages. Here are the different message types: https://python.langchain.com/docs/concepts/messages/

In [ ]:
class State(BaseModel):
    """State for message-based conversation with an LLM."""
    messages: List[BaseMessage] = Field(
        default_factory=list,
        description="List of messages in the conversation."
    )

Next, we will create a connection to our model.

In [ ]:
# Initialize the chat model.
model = init_chat_model(
    os.environ["LANGCHAIN_CHAT_MODEL_OLLAMA"]
)

With the model, we can define a node for our graph.

In [ ]:
# Create the chat node.
def chat_with_llm(state: State) -> dict:
    """Process messages with the LLM and return response."""
    # Get the LLM response
    response = model.invoke(state.messages)
    
    # Append the AI response to the messages
    return {
        "messages": state.messages + [response]
    }

And with the node, we can create the graph.

In [ ]:
# Create the graph builder.
builder = StateGraph(State)

# Add edges - simple linear flow
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

builder.add_node("chat", chat_with_llm)

# Compile the graph.
graph = builder.compile()

# Render the graph visualization.
display(graph)

## Example 1: Simple question

In [ ]:
result = graph.invoke(
    State(messages=[
        HumanMessage(content="What is the capital of Germany?")
    ])
)
for message in result["messages"]:
    message.pretty_print()

## Example 2: Multi-turn conversation

In [ ]:
# First turn
conversation_state = State(messages=[
    HumanMessage(content="Hello! My name is Alice.")
])
result = graph.invoke(conversation_state)

for message in result["messages"]:
    message.pretty_print()
print()

# Second turn - continue the conversation
# Create a new state with all messages from the result
conversation_state = State(**result)
conversation_state.messages.append(
    HumanMessage(content="What is my name?")
)
result = graph.invoke(conversation_state)

for message in result["messages"]:
    message.pretty_print()
print()


LangGraph provides the checkpointer functionality to maintain the state of the conversation across multiple turns. This allows for a more coherent and context-aware interaction with the model. See: https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpoints

## Using a reducer for the messages

Instead of adding each message to the state individually, we can use a reducer function to manage the messages more effectively. This approach allows us to define how messages are added or updated in the state in a more declarative manner.

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages

class State(BaseModel):
    """State for message-based conversation with an LLM."""
    messages: Annotated[List[BaseMessage], add_messages] = Field( # This is new! add_messages is key.
        default_factory=list,
        description="List of messages in the conversation."
    )

# Create the chat node.
def chat_with_llm(state: State) -> dict:
    """Process messages with the LLM and return response."""
    # Get the LLM response
    response = model.invoke(state.messages)
    
    # Append the AI response to the messages
    return {
        "messages": [response]
    }

# Invoke the graph again with the new State definition.
print("--- Example 1: Simple Question ---")
result = graph.invoke(
    State(messages=[
        HumanMessage(content="What is the capital of Germany?")
    ])
)
for message in result["messages"]:
    message.pretty_print()

# Done.